# Breast Cancer PubMed Paper Fetcher

Fetches PubMed abstracts and PMC full-text articles for all unique genes across BC gene-set communities and stores them in a single unified BC literature database.

In [ ]:
import os
import re
import time
import requests
import xml.etree.ElementTree as ET

from typing import Dict, List, Any, Optional
from datetime import datetime

import pandas as pd
from tqdm import tqdm

In [ ]:
# =============================================================================
# ENVIRONMENT VARIABLES
# =============================================================================

# Set these once in Jupyter before running:
#
#import os
#os.environ["NCBI_EMAIL"] = "email here"
#os.environ["NCBI_API_KEY"] = "api here"


# =============================================================================
# AML SEARCH TERMS
# =============================================================================

BREAST_CANCER_TERMS = (
    '"Breast Neoplasms"[MeSH Terms] OR '
    '"breast cancer"[Title/Abstract] OR '
    '"breast cancer"[All Fields] OR '
    '"breast carcinoma"[Title/Abstract] OR '
    '"breast carcinoma"[All Fields] OR '
    '"mammary carcinoma"[Title/Abstract] OR '
    '"mammary carcinoma"[All Fields] OR '
    '"breast tumor"[Title/Abstract] OR '
    '"breast tumour"[Title/Abstract] OR '
    '"breast neoplasm"[Title/Abstract] OR '
    '("BC"[Title/Abstract] AND "breast"[All Fields]) OR '
    '"triple negative breast cancer"[Title/Abstract] OR '
    '"TNBC"[Title/Abstract] OR '
    '"HER2-positive breast cancer"[Title/Abstract] OR '
    '"luminal breast cancer"[Title/Abstract]'
)


# =============================================================================
# UNIQUE GENE EXTRACTION
# =============================================================================

def extract_unique_genes_with_communities(
    gene_sets: Dict[str, List[str]]
) -> tuple:

    gene_to_communities: Dict[str, List[str]] = {}

    for community, genes in gene_sets.items():
        for gene in genes:
            gene_to_communities.setdefault(gene, []).append(community)

    unique_genes = list(gene_to_communities.keys())

    print(
        f"Extracted {len(unique_genes)} unique genes "
        f"from {len(gene_sets)} communities"
    )

    return unique_genes, gene_to_communities


# =============================================================================
# PMC FULL-TEXT FETCHER
# =============================================================================

def fetch_pmc_full_text(
    pmcid: str,
    email: str,
    api_key: Optional[str] = None,
) -> Optional[str]:

    try:

        params = {
            "db": "pmc",
            "id": pmcid,
            "rettype": "full",
            "retmode": "xml",
            "email": email,
            "tool": "AML_GeneResearch",
        }

        if api_key:
            params["api_key"] = api_key

        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params=params,
            headers={"User-Agent": f"AML_GeneResearch/1.0 (mailto:{email})"},
            timeout=30,
        )

        if not response.ok:
            return None

        root = ET.fromstring(response.content)

        body = root.find(".//body")

        if body is None:
            return None

        paragraphs = []

        for elem in body.iter():
            if elem.tag in {"p", "title", "sec"}:
                text = "".join(elem.itertext()).strip()

                if text:
                    paragraphs.append(text)

        full_text = "\n\n".join(paragraphs)

        return full_text if full_text else None

    except Exception:
        return None


# =============================================================================
# PUBMED XML PARSER
# =============================================================================

def parse_pubmed_xml(xml_content: str) -> List[Dict[str, Any]]:

    papers = []

    try:

        root = ET.fromstring(xml_content)

        for article_elem in root.findall(".//PubmedArticle"):

            try:

                paper: Dict[str, Any] = {}

                # PMID
                pmid_elem = article_elem.find(".//PMID")
                paper["pmid"] = (
                    pmid_elem.text if pmid_elem is not None else "Unknown"
                )

                # Title
                title_elem = article_elem.find(".//ArticleTitle")
                paper["title"] = (
                    title_elem.text
                    if title_elem is not None
                    else "Unknown Title"
                )

                # Abstract
                abstract_parts = article_elem.findall(".//AbstractText")

                abstract_text = " ".join(
                    [p.text for p in abstract_parts if p.text]
                )

                paper["abstract"] = (
                    abstract_text
                    if abstract_text
                    else "Abstract not available"
                )

                # Journal
                journal_elem = article_elem.find(".//Journal/Title")

                paper["journal"] = (
                    journal_elem.text
                    if journal_elem is not None
                    else "Unknown Journal"
                )

                # ISSN
                issn_elem = article_elem.find(
                    ".//Journal/ISSN[@IssnType='Print']"
                )

                eissn_elem = article_elem.find(
                    ".//Journal/ISSN[@IssnType='Electronic']"
                )

                paper["issn"] = (
                    issn_elem.text if issn_elem is not None else ""
                )

                paper["eissn"] = (
                    eissn_elem.text if eissn_elem is not None else ""
                )

                # Year
                pub_date = article_elem.find(".//PubDate/Year")

                if pub_date is not None:

                    paper["year"] = pub_date.text

                else:

                    medline_date = article_elem.find(".//PubDate/MedlineDate")

                    if medline_date is not None and medline_date.text:

                        year_match = re.search(
                            r"\d{4}",
                            medline_date.text
                        )

                        paper["year"] = (
                            year_match.group(0)
                            if year_match
                            else "Unknown"
                        )

                    else:

                        paper["year"] = "Unknown"

                # Volume
                volume_elem = article_elem.find(".//Volume")

                paper["volume"] = (
                    volume_elem.text
                    if volume_elem is not None
                    else "Unknown"
                )

                # Pages
                pages_elem = article_elem.find(".//MedlinePgn")

                paper["pages"] = (
                    pages_elem.text
                    if pages_elem is not None
                    else "Unknown"
                )

                # Authors
                authors = []

                author_list = article_elem.find(".//AuthorList")

                if author_list is not None:

                    for author in author_list.findall(".//Author"):

                        last = author.find("LastName")
                        inits = author.find("Initials")

                        if last is not None and inits is not None:
                            authors.append(f"{last.text} {inits.text}")

                        elif last is not None:
                            authors.append(last.text)

                paper["authors"] = (
                    ", ".join(authors)
                    if authors
                    else "Unknown Authors"
                )

                # DOI + PMCID
                doi = None
                pmc_id = None

                id_list = article_elem.find(".//ArticleIdList")

                if id_list is not None:

                    for id_elem in id_list.findall(".//ArticleId"):

                        if id_elem.get("IdType") == "doi":
                            doi = id_elem.text

                        elif id_elem.get("IdType") == "pmc":
                            pmc_id = id_elem.text

                paper["doi"] = doi or "DOI not available"
                paper["pmcid"] = pmc_id

                papers.append(paper)

            except Exception:
                continue

    except Exception:
        pass

    return papers


# =============================================================================
# PUBMED FETCH FOR SINGLE GENE
# =============================================================================

def fetch_pubmed_for_gene(
    gene: str,
    start_year: int = 2010,
    end_year: int = 2026,
    max_results: int = 10,
) -> List[Dict[str, Any]]:

    email = os.environ.get("NCBI_EMAIL")
    api_key = os.environ.get("NCBI_API_KEY")

    if not email:
        raise ValueError("NCBI_EMAIL environment variable not set")

    try:

        gene_terms = (
            f'("{gene}"[Title/Abstract] OR "{gene}"[All Fields])'
        )

        query = (
            f"({gene_terms} AND ({BREAST_CANCER_TERMS})) "
            f"AND ({start_year}:{end_year}[PDAT])"
        )

        headers = {
            "User-Agent": f"AML_GeneResearch/1.0 (mailto:{email})"
        }

        # ---------------------------------------------------------------------
        # ESEARCH
        # ---------------------------------------------------------------------

        search_params = {
            "db": "pubmed",
            "term": query,
            "retmax": max_results,
            "sort": "relevance",
            "email": email,
            "tool": "AML_GeneResearch",
        }

        if api_key:
            search_params["api_key"] = api_key

        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params=search_params,
            headers=headers,
            timeout=30,
        )

        if not response.ok:
            return []

        root = ET.fromstring(response.text)

        pmids = [elem.text for elem in root.findall(".//Id")]

        if not pmids:
            return []

        # ---------------------------------------------------------------------
        # EFETCH
        # ---------------------------------------------------------------------

        fetch_params = {
            "db": "pubmed",
            "id": ",".join(pmids),
            "retmode": "xml",
            "email": email,
            "tool": "AML_GeneResearch",
        }

        if api_key:
            fetch_params["api_key"] = api_key

        fetch_response = requests.post(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            data=fetch_params,
            headers=headers,
            timeout=60,
        )

        if not fetch_response.ok:
            return []

        papers = parse_pubmed_xml(fetch_response.text)

        # ---------------------------------------------------------------------
        # PMC FULL TEXT
        # ---------------------------------------------------------------------

        for paper in papers:

            pmcid = paper.get("pmcid")

            if pmcid:

                time.sleep(0.35)

                full_text = fetch_pmc_full_text(
                    pmcid,
                    email,
                    api_key,
                )

                paper["full_text"] = (
                    full_text
                    if full_text
                    else "Full text not available via PMC"
                )

            else:

                paper["full_text"] = (
                    "Full text not available via PMC"
                )

        return papers

    except Exception as e:

        print(f"fetch_pubmed_for_gene({gene}) failed: {e}")

        return []


# =============================================================================
# FETCH FOR ALL UNIQUE GENES
# =============================================================================

def fetch_pubmed_for_all_unique_genes(
    gene_sets: Dict[str, List[str]],
    start_year: int = 2010,
    end_year: int = 2026,
    max_results_per_gene: int = 10,
) -> pd.DataFrame:

    unique_genes, gene_to_communities = (
        extract_unique_genes_with_communities(gene_sets)
    )

    all_results = []
    genes_with_papers = 0

    for gene in tqdm(
        unique_genes,
        desc="Genes",
        unit="gene",
        ncols=100,
    ):

        try:

            papers = fetch_pubmed_for_gene(
                gene=gene,
                start_year=start_year,
                end_year=end_year,
                max_results=max_results_per_gene,
            )

            if papers:

                genes_with_papers += 1

                for paper in papers:

                    paper["gene"] = gene

                    paper["communities"] = ", ".join(
                        gene_to_communities[gene]
                    )

                    paper["community_count"] = len(
                        gene_to_communities[gene]
                    )

                    all_results.append(paper)

            time.sleep(0.35)

        except Exception as e:

            print(f"Error processing gene {gene}: {e}")

            continue

    total = len(unique_genes)

    print(
        f"\nGenes with papers: "
        f"{genes_with_papers}/{total} "
        f"({genes_with_papers/total*100:.1f}%)"
    )

    print(f"Total papers: {len(all_results)}")

    if not all_results:
        return pd.DataFrame()

    df = pd.DataFrame(all_results)

    column_order = [
        "gene",
        "communities",
        "community_count",
        "title",
        "authors",
        "journal",
        "year",
        "volume",
        "pages",
        "doi",
        "pmid",
        "pmcid",
        "abstract",
        "full_text",
    ]

    df = df[[col for col in column_order if col in df.columns]]

    if "pmid" in df.columns and "gene" in df.columns:

        before = len(df)

        df = df.drop_duplicates(subset=["pmid", "gene"])

        dropped = before - len(df)

        if dropped:
            print(f"Dropped {dropped} duplicate rows")

    return df


# =============================================================================
# ENTRY POINT
# =============================================================================

ANNOTATION_CSV = "Breast Cancer Annotation.csv"
OUTPUT_CSV = "BC_Paper_DB.csv"

START_YEAR = 2010
END_YEAR = 2026
MAX_RESULTS_PER_GENE = 10


# =============================================================================
# LOAD INPUT DATA
# =============================================================================

annotation_df = pd.read_csv(ANNOTATION_CSV)

annotation_df.columns = [c.strip() for c in annotation_df.columns]

print(f"Loaded {len(annotation_df)} rows")


gene_col = next(
    (
        c for c in annotation_df.columns
        if c in (
            "Genes_String",
            "Contributing_Genes",
            "Genes",
            "Final_Contributing_Genes",
        )
    ),
    None,
)

community_col = next(
    (
        c for c in annotation_df.columns
        if c in (
            "Community",
            "Set_ID",
            "community",
            "index",
        )
    ),
    None,
)

if gene_col is None or community_col is None:
    raise ValueError(
        f"Could not find required columns.\n"
        f"Columns: {list(annotation_df.columns)}"
    )

print(f"Using gene column: {gene_col}")
print(f"Using community column: {community_col}")


# =============================================================================
# BUILD GENE SETS
# =============================================================================

gene_sets: Dict[str, List[str]] = {}

for _, row in annotation_df.iterrows():

    community = str(row[community_col])

    raw_genes = str(row[gene_col])

    if raw_genes and raw_genes.lower() != "nan":

        sep = "," if "," in raw_genes else ";"

        genes = [
            g.strip()
            for g in raw_genes.split(sep)
            if g.strip()
        ]

        gene_sets[community] = genes

print(f"Built {len(gene_sets)} gene-set communities")


# =============================================================================
# RESUME SUPPORT
# =============================================================================

already_fetched = set()

if os.path.exists(OUTPUT_CSV):

    existing_df = pd.read_csv(OUTPUT_CSV)

    already_fetched = set(
        existing_df["gene"].dropna().unique()
    )

    print(
        f"Resuming from existing database "
        f"({len(already_fetched)} genes already fetched)"
    )

    gene_sets_filtered = {}

    for community, genes in gene_sets.items():

        remaining = [
            g for g in genes
            if g not in already_fetched
        ]

        if remaining:
            gene_sets_filtered[community] = remaining

    gene_sets = gene_sets_filtered


# =============================================================================
# RUN RETRIEVAL
# =============================================================================

papers_df = fetch_pubmed_for_all_unique_genes(
    gene_sets=gene_sets,
    start_year=START_YEAR,
    end_year=END_YEAR,
    max_results_per_gene=MAX_RESULTS_PER_GENE,
)


# =============================================================================
# SAVE OUTPUT
# =============================================================================

if not papers_df.empty:

    if os.path.exists(OUTPUT_CSV) and already_fetched:

        papers_df.to_csv(
            OUTPUT_CSV,
            mode="a",
            header=False,
            index=False,
        )

        print(
            f"\nAppended {len(papers_df)} rows to {OUTPUT_CSV}"
        )

    else:

        papers_df.to_csv(
            OUTPUT_CSV,
            index=False,
        )

        print(
            f"\nSaved {len(papers_df)} rows to {OUTPUT_CSV}"
        )

else:

    print("\nNo papers retrieved")


print(f"\nDone! AML database saved to: {OUTPUT_CSV}")